In [ ]:
# Imports
import os, re, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# ======================== CONFIG ========================
DATA_PATH = "final_neural_network_data.csv"
TREATMENT_PATH = "treatments.csv"
MODEL_SAVE_PATH = "best_disease_model.h5"

MAX_WORDS = 5000
MAX_LEN = 100
EMBED_DIM = 64
LSTM_UNITS = 128
DROPOUT = 0.3
SEED = 42

EPOCHS_STAGE1 = 50
EPOCHS_STAGE2 = 30
BATCH_SIZE = 16
LEARNING_RATE_STAGE1 = 0.001

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ======================== DATA LOADING ========================
df = pd.read_csv(DATA_PATH).dropna(subset=["Disease", "Description"])
treatments_df = pd.read_csv(TREATMENT_PATH)[["Disease", "Treatment", "Avoid"]].dropna()

df["Disease"] = df["Disease"].str.strip()
treatments_df["Disease"] = treatments_df["Disease"].str.strip()

# ======================== PREPROCESS ========================
STOP_WORDS = set(stopwords.words("english"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return " ".join(tokens)

df["clean_text"] = df["Description"].apply(preprocess)

# ======================== ENCODING ========================
le = LabelEncoder()
df["label"] = le.fit_transform(df["Disease"])
NUM_CLASSES = len(le.classes_)

X = df["clean_text"].values
y = df["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

# ======================== TOKEN ========================
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

def encode(texts):
    return pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=MAX_LEN)

X_train_enc = encode(X_train)
X_val_enc = encode(X_val)
X_test_enc = encode(X_test)

y_train_cat = to_categorical(y_train)
y_val_cat = to_categorical(y_val)
y_test_cat = to_categorical(y_test)

# ======================== MODEL ========================
def build_model(lr):
    inp = Input(shape=(MAX_LEN,))
    x = Embedding(MAX_WORDS, EMBED_DIM)(inp)
    x = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True))(x)
    x = Dropout(DROPOUT)(x)
    x = Bidirectional(LSTM(LSTM_UNITS // 2))(x)
    x = BatchNormalization()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(DROPOUT)(x)
    out = Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

model = build_model(LEARNING_RATE_STAGE1)

model.summary()

# ======================== TRAIN ========================
callbacks = [
    ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True, monitor='val_accuracy', mode='max'),
    EarlyStopping(patience=5, restore_best_weights=True)
]

model.fit(X_train_enc, y_train_cat,
          validation_data=(X_val_enc, y_val_cat),
          epochs=EPOCHS_STAGE1,
          batch_size=BATCH_SIZE,
          callbacks=callbacks)

# ======================== LOAD BEST ========================
best_model = load_model(MODEL_SAVE_PATH)

# ======================== EVAL ========================
loss, acc = best_model.evaluate(X_test_enc, y_test_cat)
print("\nTest Accuracy:", acc)

y_pred = np.argmax(best_model.predict(X_test_enc), axis=1)
print(classification_report(y_test, y_pred))

# ======================== TREATMENT ========================
treatment_map = {
    row["Disease"].lower(): {
        "treatment": row["Treatment"],
        "avoid": row["Avoid"]
    }
    for _, row in treatments_df.iterrows()
}

# ======================== PREDICT ========================
def predict_disease(text, top_k=3):
    clean = preprocess(text)
    encoded = encode([clean])
    probs = best_model.predict(encoded, verbose=0)[0]

    top_idx = np.argmax(probs)
    disease = le.classes_[top_idx]
    confidence = probs[top_idx] * 100

    top_k_idx = np.argsort(probs)[::-1][:top_k]
    top_preds = [
        {"disease": le.classes_[i], "confidence": f"{probs[i]*100:.1f}%"}
        for i in top_k_idx
    ]

    tokens = clean.split()[:10]
    why_text = f"Matched words: {', '.join(tokens)} → '{disease}' ({confidence:.1f}%)"

    info = treatment_map.get(disease.lower(), {})

    return {
        "disease": disease,
        "confidence": f"{confidence:.1f}%",
        "why": why_text,
        "treatment": info.get("treatment", "Consult doctor"),
        "avoid": info.get("avoid", "N/A"),
        "top_k": top_preds
    }

# ======================== AUTO TEST ========================
print("\n" + "="*50)
print("AUTO TEST MODE")
print("="*50)

samples = [
    "red itchy skin with white patches",
    "severe headache and sensitivity to light",
    "joint pain and swelling in knees"
]

for s in samples:
    r = predict_disease(s)

    print("\n" + "-"*40)
    print(f"Input: {s}")
    print(f"Disease: {r['disease']} ({r['confidence']})")
    print(f"Why: {r['why']}")
    print(f"Treatment: {r['treatment']}")
    print(f"Avoid: {r['avoid']}")

    top3 = [f"{p['disease']} ({p['confidence']})" for p in r["top_k"]]
    print(f"Top-3: {' | '.join(top3)}")
    print("-"*40)

print("\nModel saved to:", MODEL_SAVE_PATH)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 100, 256)       │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 691,272 (2.64 MB)

 Trainable params: 691,016 (2.64 MB)

 Non-trainable params: 256 (1.00 KB)

...

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.7917 - loss: 1.0031 - val_accuracy: 0.3333 - val_loss: 2.0284
Epoch 6/50
4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8958 - loss: 0.7002

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.8750 - loss: 0.6615 - val_accuracy: 0.5556 - val_loss: 1.9744
Epoch 7/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.9444 - loss: 0.3995 - val_accuracy: 0.5556 - val_loss: 1.9413
Epoch 8/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9861 - loss: 0.2665 - val_accuracy: 0.4444 - val_loss: 1.8805
Epoch 9/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.1387 - val_accuracy: 0.3333 - val_loss: 1.8581
Epoch 10/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 0.0840 - val_accuracy: 0.3333 - val_loss: 1.9285
Epoch 11/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 0.0684 - val_accuracy: 0.5556 - val_loss: 1.8935
Epoch 12/50
4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.0471

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 1.0000 - loss: 0.0447 - val_accuracy: 0.6667 - val_loss: 1.8169
Epoch 13/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 1.0000 - loss: 0.0505 - val_accuracy: 0.6667 - val_loss: 1.7768
Epoch 14/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.0244 - val_accuracy: 0.6667 - val_loss: 1.7592
Epoch 15/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 0.0288 - val_accuracy: 0.6667 - val_loss: 1.7734
Epoch 16/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 1.0000 - loss: 0.0240 - val_accuracy: 0.4444 - val_loss: 1.8157
Epoch 17/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 1.0000 - loss: 0.0323 - val_accuracy: 0.3333 - val_loss: 1.7795
Epoch 18/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 1.0000 - loss: 0.0273 - val_accuracy: 0.3333 - val_loss: 1.7477
Epoch 19/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 1.0000 - loss: 0.0273 - val_accuracy: 0.3333 - val_loss: 1.7028
Epoc

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 1.0000 - loss: 0.0173 - val_accuracy: 0.7778 - val_loss: 1.5560
Epoch 27/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.0088 - val_accuracy: 0.7778 - val_loss: 1.5056
Epoch 28/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 1.0000 - loss: 0.0094 - val_accuracy: 0.7778 - val_loss: 1.4974
Epoch 29/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 0.0078 - val_accuracy: 0.4444 - val_loss: 1.5734
Epoch 30/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 1.0000 - loss: 0.0193 - val_accuracy: 0.4444 - val_loss: 1.6068
Epoch 31/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 1.0000 - loss: 0.0081 - val_accuracy: 0.3333 - val_loss: 1.6598
Epoch 32/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 0.0050 - val_accuracy: 0.3333 - val_loss: 1.6431
Epoch 33/50
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.0089 - val_accuracy: 0.3333 - val_loss: 1.6095


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1000ms/step - accuracy: 0.3000 - loss: 1.9085

Test Accuracy: 0.30000001192092896
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 456ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         2
           3       0.00      0.00      0.00         2
           4       1.00      1.00      1.00         1
           5       1.00      1.00      1.00         1
           6       1.00      0.50      0.67         2
           7       0.00      0.00      0.00         1

    accuracy                           0.30        10
   macro avg       0.38      0.31      0.33        10
weighted avg       0.40      0.30      0.33        10


AUTO TEST MODE

----------------------------------------
Input: red itchy skin with white patches
Disease: Vitiligo (19.7%)
Why: Matched words: red, itchy, skin, white, patches → 'Vitiligo' (19.7%)
T